# 00 · Setup check

**Goal.** Make sure the environment, folders and data are in place before any real work.

**Inputs.** Nothing except `config.py`.
**Outputs.** Nothing saved. Only printed checks.

**Steps**
1. Load the project settings
2. Check the Python packages
3. Check the folders and free space
4. Check which raw data are already downloaded

Every notebook in this project follows the same layout: a short goal at the top, numbered steps, one idea per code cell, and a **What to check** line after each result.

## Step 1 · Load the project settings

All paths and study settings live in `config.py`, so we import them here instead of typing paths in every notebook. The first lines add the repo folder to Python's search path, so `config` and `src` can be found from inside `notebooks/`.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from config import *
from src.flood_utils import rmse, skill_vs

print("repo folder :", REPO_DIR)
print("data folder :", DATA_DIR)
print("years       :", YEARS, "| flood window:", FLOOD_WINDOW)

repo folder : /home/tb3203/thailand-flood-ml
data folder : /data9/CLMT5120/work/tb3203/flood_data
years       : [2024, 2025, 2026] | flood window: ('08-04', '10-21')


**What to check:** the repo folder is your home copy of the repo, and the data folder is on `/data9`.

## Step 2 · Check the Python packages

If any line says MISSING, install it into the project environment (not the base one) from a terminal:
`source ~/flood-env/bin/activate && pip install <name>`

In [2]:
import importlib

packages = ["numpy", "pandas", "xarray", "rasterio", "rioxarray", "geopandas",
            "shapely", "pyproj", "sklearn", "scipy", "matplotlib", "pyarrow"]
for name in packages:
    try:
        mod = importlib.import_module(name)
        print(f"{name:12s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"{name:12s} MISSING")

print("\nPython:", sys.version.split()[0], "| running from:", sys.executable)

numpy        2.4.6
pandas       3.1.0
xarray       2026.9.0
rasterio     1.4.4
rioxarray    0.19.0
geopandas    1.2.0
shapely      2.2.0
pyproj       3.7.2
sklearn      1.9.1
scipy        1.17.1
matplotlib   3.11.2
pyarrow      26.0.1

Python: 3.11.14 | running from: /home/tb3203/flood-env/bin/python


**What to check:** nothing is MISSING, scikit-learn is 1.4 or newer, and Python runs from `flood-env`.

## Step 3 · Check the folders and free space

The home folder (10 GB, backed up) holds the code. The work folder (50 GB) holds the raw data. We print how much each one uses.

In [3]:
import shutil, subprocess

def folder_size(p):
    out = subprocess.run(["du", "-sh", str(p)], capture_output=True, text=True).stdout.split()
    return out[0] if out else "not found"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("repo (home)  :", folder_size(REPO_DIR))
print("whole home   :", folder_size(Path.home()), "of 10 GB")
print("data (/data9):", folder_size(DATA_DIR))
free = shutil.disk_usage(DATA_DIR).free / 1e9
print(f"free on the data disk: {free:.0f} GB")

repo (home)  : 9.5M
whole home   : 6.2G of 10 GB
data (/data9): 4.0G
free on the data disk: 17729 GB


**What to check:** home stays well under 10 GB. If it doesn't, the raw data are still in home and should be moved to `/data9`.

## Step 4 · Check which raw data are already downloaded

This counts files in each raw-data folder. Zero means that dataset still needs notebook 01 (or moving from the old project folder).

In [4]:
checks = {
    "GISTDA flood maps (.shp)": (RAW_GISTDA, "**/*.shp", 376),
    "CHIRPS daily rain (.tif)": (RAW_CHIRPS, "*.tif", 459),
    "HydroSHEDS (.shp)":        (RAW_HYDRO, "**/*.shp", None),
}
for label, (folder, pattern, expected) in checks.items():
    n = len(list(folder.glob(pattern))) if folder.exists() else 0
    note = "" if expected is None else f"(expect {expected})"
    print(f"{label:28s} {n:5d} {note}")

GISTDA flood maps (.shp)       377 (expect 376)
CHIRPS daily rain (.tif)       459 (expect 459)
HydroSHEDS (.shp)                0 


**What to check:** 376 flood maps and 459 rain files. HydroSHEDS can be zero until step 1 of the work plan is done.

## Summary

If all four steps look right, the environment is ready. Next: `01_download_data.ipynb`.